# Gemma-12B: Neutral-topic follow-up pilot
**25 posts (not 50/100): this is a scoped pilot, not a full replication. Gemma-12B only, per the
supervisor's request -- not rolled out to the other 5 models.**

In [1]:
import sys, subprocess

# 1. Uninstall torchaudio
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

# 2. Install PyTorch with CUDA 12.4
subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

# 3. Install latest transformers and accelerate (allowing pip to pull compatible tokenizers naturally)
subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

print("✅ Installation complete — restart the kernel now")



ERROR: Will not install to the user site because it will lack sys.path precedence to torch in /usr/local/jupyterhub-2026/lib64/python3.13/site-packages


CalledProcessError: Command '['/usr/local/jupyterhub-2026/bin/python3.13', '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0', '--index-url', 'https://download.pytorch.org/whl/cu124', '--user', '-q']' returned non-zero exit status 1.

Restart kernel after running the setup cell above.

In [ ]:
!nvidia-smi

In [ ]:
# --- HF Auth ---
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

# --- Path setup ---
from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

# --- Load Gemma model ---
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/gemma-4-12B-it"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, padding_side="left")
device = model.device

In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: neutral-topic follow-up pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_neutral/  -- Gemma-12B only for this pilot
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "neutral_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2), and the one the climate-pilot inversion was found under
correct_base = ROOT_DIR / "neutral_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "neutral_pilot/posts/incorrect/PNGs/metrics/realistic"


In [ ]:
from e1_utils.inference_gemma import run_inference_gemma

In [ ]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_gemma import run_inference_with_scores_gemma

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [ ]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_gemma)

In [ ]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_gemma)

In [ ]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")

## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [ ]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_gemma)

In [ ]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_gemma)

In [ ]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")

## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_gemma)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

SCALE_VALUES = [0, 10, 100, 1000, 10000, 100000, 1000000]
SCALE_LABELS = ["0", "10", "100", "1K", "10K", "100K", "1M"]

# Same red, gray, blue palette already used for the authority grid
RED_GRAY_BLUE = mcolors.LinearSegmentedColormap.from_list(
    "red_gray_blue", ["#e34948", "#f0efec", "#2a78d6"]
)

def plot_metrics_paired_grid(output_dir: Path, filename: str, title: str,
                              position_gap_threshold: float = 60.0, save: bool = True):
    data = json.loads((Path(output_dir) / filename).read_text())
    valid = [r for r in data if r["liked_variant"] in ("correct", "incorrect")]
    n = len(SCALE_VALUES)
    pct = [[None] * n for _ in range(n)]
    gap = [[None] * n for _ in range(n)]

    for i, inc_s in enumerate(SCALE_VALUES):
        for j, cor_s in enumerate(SCALE_VALUES):
            cell = [r for r in valid if r["correct_scale"] == cor_s and r["incorrect_scale"] == inc_s]
            if not cell:
                continue
            pct[i][j] = sum(1 for r in cell if r["liked_variant"] == "correct") / len(cell) * 100
            halves = []
            for slot in ("A", "B"):
                half = [r for r in cell if (r["post_a_variant"] == "correct") == (slot == "A")]
                if half:
                    halves.append(sum(1 for r in half if r["liked_variant"] == "correct") / len(half) * 100)
            gap[i][j] = abs(halves[0] - halves[1]) if len(halves) == 2 else None

    grid = np.array([[np.nan if v is None else v for v in row] for row in pct])
    fig, ax = plt.subplots(figsize=(9, 7.5))
    im = ax.imshow(grid, cmap=RED_GRAY_BLUE, vmin=0, vmax=100, aspect="auto")

    for i in range(n):
        for j in range(n):
            if np.isnan(grid[i][j]):
                continue
            suspect = gap[i][j] is not None and gap[i][j] >= position_gap_threshold
            ax.text(j, i, f"{grid[i][j]:.0f}%", ha="center", va="center", fontsize=10, fontweight="bold", color="black")

    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(SCALE_LABELS); ax.set_yticklabels(SCALE_LABELS)
    ax.invert_yaxis()
    ax.set_xlabel("Reactions on the correct post", fontsize=16)
    ax.set_ylabel("Reactions on the incorrect post", fontsize=16)
    ax.set_title(title, fontsize=13, fontweight="bold")
    fig.colorbar(im, ax=ax, label="Chose the correct post (%), gray at 50% is chance")
    plt.tight_layout()

    if save:
        out = Path(output_dir) / f"{Path(filename).stem}_grid.png"
        plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
plot_metrics_paired_grid(OUTPUT_DIR, "e1_results_metrics_paired.json",
                          title="Gemma-12B, neutral topic pilot: chose the correct post (%)")

In [ ]:
# however, how much of this behaviour is positional bias?
import json
from pathlib import Path


def check_position_bias(output_dir: Path, filename: str):
    """Check whether Gemma-12B's answers depend on which slot (A or B) the
    correct post landed in, rather than on the post itself.

    Two checks, matching the ones already used in the climate pilot write up.

    1. Tied engagement default. At tied engagement (correct_scale equals
       incorrect_scale) both posts are equally credible, so a model with no
       slot preference should answer A about half the time. A rate far from
       50% means the model is defaulting to a slot rather than judging the
       content.

    2. Disadvantaged correct post, split by slot. Among the off diagonal
       trials where the correct post has fewer reactions than the incorrect
       post, accuracy is computed separately for trials where the correct
       post sat in slot A and trials where it sat in slot B. If the model
       were reading content rather than position, these two numbers should
       be close together. A large gap between them means part of what looks
       like an engagement effect is really a position effect.
    """
    data = json.loads((Path(output_dir) / filename).read_text())
    valid = [r for r in data if r["answer"] in ("A", "B")]

    diag = [r for r in valid if r["correct_scale"] == r["incorrect_scale"]]
    a_rate = sum(1 for r in diag if r["answer"] == "A") / len(diag) * 100
    print(f"Tied engagement trials: n={len(diag)}")
    print(f"  Answered 'A': {a_rate:.1f}% (50% would mean no slot default)")

    disadvantaged = [
        r for r in valid
        if r["correct_scale"] != r["incorrect_scale"]
        and r["correct_scale"] < r["incorrect_scale"]
    ]
    print(f"\nTrials where the correct post has fewer reactions: n={len(disadvantaged)}")

    slot_accuracy = {}
    for slot in ("A", "B"):
        in_slot = [r for r in disadvantaged if (r["post_a_variant"] == "correct") == (slot == "A")]
        if not in_slot:
            continue
        acc = sum(1 for r in in_slot if r["liked_variant"] == "correct") / len(in_slot) * 100
        slot_accuracy[slot] = acc
        print(f"  Correct post in slot {slot} (n={len(in_slot)}): chose correctly {acc:.1f}% of the time")

    if len(slot_accuracy) == 2:
        gap = abs(slot_accuracy["A"] - slot_accuracy["B"])
        print(f"\nGap between the two slots: {gap:.1f} points")
        if gap >= 20:
            print("  This gap is large enough that some of the apparent engagement effect")
            print("  is likely coming from which slot the post was shown in, not just from")
            print("  how many reactions it had.")
        else:
            print("  This gap is small, so the result is unlikely to be driven by position.")


if __name__ == "__main__":
    OUTPUT_DIR = Path("outputs")
    check_position_bias(OUTPUT_DIR, "e1_results_metrics_paired.json")